# Глава 13. Интерпретируемость

## Введение
Взрывной рост качества языковых моделей после 2022 года оказался неожиданным. Создали моделей неоднократно говорили, что открывшиеся возможности стали для них сюрпризом.

> "Мы не понимаем, как работают большие языковые модели (LLM). Мы их построили, но не можем объяснить, почему они делают то, что делают. Внутреннее пространство модели — это чёрный ящик возникающих структур и поведения"
> (Андрей Карпаты, сооснователь OpenAI)

В индустрии фразу "мы не понимаем, как это работает" так часто повторяли, что она стала хрестоматийной. Я бы внес небольшую корректировку - мы конечно понимаем, *как* модель работает, в том смысле, что мы можем проследить, как распространяется сигнал, вплоть до отдельного нейрона. Скорее мы не понимаем, *почему* это так хорошо работает, почему это так слажено работает в системе, почему так дёшево получить ненаблюлаеме ранее навыки? В данной главе посмотреим, как можно ответить и на первый вопрос "Как?", и попробуем проложить дорогу к ответу на второй.

__Интерпретируемость__ - свойство модели быть прозрачной и понятной. Это понятие из классического машинного обучения, оно стало появляться ещё в 1970 годах в контексте первых автоматизированных систем принятия решений. Для многих приложений оказалось важно не только, какое решение принято, но и почему оно принято. Для ряда индустрий (банковской, финансовой, страховой) прозрачность стала обязательным требованием, закрепленным регламентами. Банк не может просто отказать в кредите, он должен быть готов объяснить регулятору, по каким критериям не проходит заявитель. Эти регламенты в том числе долгое время затрудняли возможность использования глубоких сетей. В итоге интерпретируемость стала значимой характеристикой наравне с точностью, размером и скоростью. 

К концу 2010-х годов оформился целый раздел исследований - *Объяснимый ИИ* (Explainable AI) — это совокупность методов и инструментов для анализа почему модель ведет себя так или по-другому. Их можно поделить на глобальные методы, оценивающие степень значимости признака для модели, и локальные, которые отвечают на вопрос, почему для входа $x$ предсказание $\hat{y}$ именно такое - какие признаки или нейроны повлияли. 

К глобальным можно отнести, например, метод перемешивания (Permutation Importance), когда значения признака случайным образом перемещиваются в выборке, а важность признака оценивается проседанием точности модели. Метод PDP кривых (Partial Dependence Plot) показывает усредненный профиль изменения предсказания при изменении анализируемого признака. Все эти подходы полезны в приложениях, но мало что говорят о внутренней логике модели.

Один из самых простых локальных методов - метод градиентных карт, описанный [(Simonyan et al., 2013)](https://arxiv.org/abs/1312.6034), он сопоставляет каждому нейрону сети градиент ошибки $\frac{\partial E}{\partial w}$. Высокое значение говорит о том, что нейрон $w$ сильно влияет на итоговый ответ, даже небольшое изменение сильно двигает ответ.

Обратное не всегда верно - не все важные нейроны демонстрируют высокую чувствительность во время предсказания. Так происходит, если, например,  модель уверена в ответе, и локальная вариация принзнаков практически не меняет ее уверенности. Чтобы более точно оценить локальный ландшафт, [(Sundararajan et al., 2017)](https://arxiv.org/abs/1703.01365) предложили сопоставлять не один градиент, а сумму градиентов, посчитанную вдоль некоторого прямого пути до точки $x$. Этот метод назвали методом интегрированных градиентов.

Метод **LIME** (Local Interpretable Model-agnostic Explanations) от [(Ribeiro et al., 2016)](https://arxiv.org/abs/1602.04938) 
решает ту же задачу - апроксимирует поведение модели в окрестности наблюдения - но делает это чуть по-другому. Для этого выполняют набор предсказаний в окрестности входа $x$ (случайно вариьруют признаки), и по этим наблюдениям восстанавливают окружающий ландшафт. Что удобно - не нужно никак учитывать формулу предсказания, её вообще не обязательно знать, модель может быть черным ящиком.

Вероятно, самый популярный метод локальной интерпретации это метод **SHAP** (SHapley Additive exPlanations). [(Lundberg & Lee, 2017)](https://arxiv.org/abs/1705.07874) позаимствовали идею из теории Шепли (Shapley Value) о построении справедливого распределения выигрыша в соотвествии с вкладом каждого участника. В задаче интерпретируемости в роли такого выигрыша выступает отклонение предсказания $f(x)$ от среднего $E[f(x)]$. Вклад признака считается как среднее изменение предсказания модели при добавлении данного признака в модель (усредненно по всем возможным комбинациям признаков). Проблема метода в том, что полный расчет с вычислением предсказаний для всех $2^N$ комбинаций признаков крайне затруднителен, поэтому в приложениях используют приближенные методы, например KernelSHAP или TreeSHAP.

### Механистическая интерпретируемость
В 2020 году Крис Ола (Crhis Olah), будущий сооснователь Antrophic, ввел в обиход термин Механическая интерпретируемость, проводя параллель с нейронаукой. Он с колегами исследовал работу нейронов сети компьютерного зрения InceptionV1 [(Olah et al., 2020)](https://distill.pub/2020/circuits/zoom-in/) и нашли среди них детекторы кривых и частей объектов. Год спустя в Anthropic [(Elhage et al., 2021)](https://transformer-circuits.pub/2021/framework/index.html) применили тот же самый подход для Трансформеров.

Название призвано подчеркнуть отличие от традиционной интерпретируемости, где в фокусе было исследования на отдельных признаков и нейронов. Тут же больший акцент делается на исследовании связок нейронов и того как они работают в системе. [(Olah, 2022)](https://transformer-circuits.pub/2022/mech-interp-essay/index.html) сравнил языковые модели со скомпилированными программами - мы ими управляем, но не понимаем, как они устроены - а механистическую интерпетацию с реверс инжинирингом, попыткой восстановить логику работы в человекочитаемом формате .

## Трансформер глазами интерпретатора
В главе 4 мы описали Трансформерную модель как стек однотипных блоков. Такую архитекутуру несложно записать в виде единого вычислительного графа, но в нем присутсвует нелинейности, которые затрудняют анализ распространения сигнала. Напоминим, как вычисляется один блок стнадартной Трансформерной модели.

$$
\begin{aligned}
\text{Самовнимание:} \quad & Q^h = x_l W_Q^h, \quad K^h = x_l W_K^h, \quad V^h = x_l W_V^h \\
& \text{head}^h = \text{softmax}\!\left(\frac{Q^h (K^h)^T}{\sqrt{d_k}}\right) V^h \\
& \text{MultiHead}(x_l) = \text{concat}(\text{head}^1, \ldots, \text{head}^H) \cdot W_O + b_O \\[6pt]
\text{Остаточная связь + Нормализация:} \quad & x' = \text{LN}(x_l + \text{MultiHead}(x_l)) \\[6pt]
\text{FFN слой:} \quad & \text{MLP}(x') = W_2 \cdot \text{ReLU}(W_1 x' + b_1) + b_2 \\[6pt]
\text{Остаточная связь + Нормализация:} \quad & x_{l+1} = \text{LN}(x' + \text{MLP}(x'))
\end{aligned}
$$

Чтобы сделать анализ более удобным [(Elhage et al., 2021)](https://transformer-circuits.pub/2021/framework/index.html) предложили упростить схему вычисления, убрав все нелинейности. Во-первых, они убрали нормализацию (LayerNorm). Она важна для обучения, но её роль для инфресена не такая большая и как правило она легко апроксимируется правильно подробанным масштабирующим коэффициентом. Обычно её просто «замораживают», беря коэффициент из реального прогона, и тогда путь от становится линейным. А [(Heimersheim, 2024)](https://arxiv.org/abs/2409.13710) показал, что GPT-2 можно дообучить вовсе без LayerNorm почти без потери качества, что удобно для экспериментов. 

Второе, что они сделали - они переписали выходную проекцию $\text{concat}(\text{head}^1, \ldots, \text{head}^H) \cdot W_O$ в виде суммы вкладов отдельных голов $\sum_{heads}$. Для этого $W_O$ на блоки строк под каждую голову, тогда компонент выходного складывается из влкадов каждой: $\text{MultiHead}(x_l) = \sum_{h=1}^{H} \text{head}^h \cdot W_O^h$. Легко видеть, что расчет стал полностью аддитивным. Такую форму записи назвали __Residual Stream__ - подчеркивая, что есть единая магистраль, по которой проходит сигнал, а в неё вливаются аддитивно сигналы от всех компонентов.
$$x^{(L)} = x^{(0)} + \sum_{l,h} \mathrm{head}_{l,h} + \sum_{l} \mathrm{MLP}_l, \qquad \text{logits} = W_U\, x^L$$

Если упростить еще и убрать FFN слой, получается Attention-only модель. Для однослойной attention-only модели с матрицей эмбедингов $W_E$ и матрицей анэмбедингов $W_U$:

$$\text{logits} = W_U (x + \sum_{h} A^h \cdot W_O^h W_V^h W_E x) = W_U W_E + \sum_{h} \underbrace{A^h}_{\text{QK-цепь}} \cdot W_U \underbrace{W_{OV}^h}_{\text{OV-цепь}} W_E$$

Получаются две независимые схемы. QK-схема решает, куда смотреть («к кому обратиться»), OV-схема — что перенести и как записать («что у него взять»). Обе матрицы низкого ранга, то есть каждая голова работает в небольшом подпространстве потока. Домножив их на эмбеддинги, получим виртуальные веса в пространстве словаря: например, у копирующей головы матрица $W_U W_{OV} W_E$ близка к диагональной — посмотрев на токен «Москва», голова повышает логит того же токена.

Без единого слоя остаётся прямой путь $W_U W_E$, и это ровно биграммная модель из главы 1. Однослойная модель добавляет к нему skip-триграммы вида «$a \dots b \to c$»: голова смотрит на ранее встреченный токен $a$ и повышает вероятность $c$ после текущего $b$. Со вторым слоем появляется композиция: головы второго слоя читают то, что записали головы первого, через запросы (Q-композиция), ключи (K-композиция) или значения (V-композиция). Самый известный пример K-композиции — индукционные головы, о которых пойдёт речь в разделе о схемах.

Полносвязный слой можно переписать как сумму вкладов отдельных нейронов:
$$\mathrm{MLP}(x) = \sum_{i} \sigma\big(k_i^\top x + b_i\big)\, v_i,$$
где $k_i$ — строка входной матрицы, а $v_i$ — столбец выходной. [(Geva et al., 2020)](https://arxiv.org/abs/2012.14913) предложили воспринимать MLP слой как память «ключ–значение»: $k_i$ задаёт паттерн, на который реагирует нейрон, а $v_i$ — то, что он добавит в поток. Если спроецировать $v_i$ через $W_U$, видно, вероятность каких токенов нейрон повышает [(Geva et al., 2022)](https://arxiv.org/abs/2203.14680).



## Признаки
В классическом машинном обучении признаки - это обычно компоненты входного вектора, в каждой из которых закодирована некотроая характеристика входа. По мере перехода на глубокие сети понятие признака расширилось и вышло за рамки входого вектора. Так, в языковых моделях признак — характеристика, которую модель кодирует во внтуренних активациях: «текст на французском», «токен внутри кавычек», «речь идёт об Эйфелевой башне» и т.п.

Признаки не гарантируют интерпретируемость. Признаки иерархичны, разные на каком слое могут располагаться в разных слоях. В языковых моделях в основном исследюутся нейроны FFN слоя. Какимим именно паттернами они кодируются?

Самое простое - предположить, что один нейрон отвечает за один признак. Иногда эта гипотеза находит подтверждение. Так [(Karpathy et al., 2015)](https://arxiv.org/abs/1506.02078) анализируя рекуррентную символьную LSTM модель, нашли конкретные измерения, отвечающие за то, открыта ли кавычка, сколько символов прошло с начала строки, кол-во пробелов и т.п. Исследователи из OpenAI [(Radford et al., 2017)](https://arxiv.org/abs/1704.01444) обучили рекуррентную модель на отзывах Amazon и обнаружили «нейрон тональности», который отвечает за общее настроение генерируемого высказывания. Искуственно меняя его значение, можно было заставить модель писать восторженные или разгромные отзывы.

Но чаще правда нейроны __полисемантичны__, то есть кодируют сразу несколько не связанных понятий. [(Olah et al., 2020)](https://distill.pub/2020/circuits/zoom-in/) когда искали паттерны в модели компьютерного зрения InceptionV1 нашли нейрон, который срабатывает одновременно на морды кошек, капоты автомобилей и кошачьи лапы. В языковых моделях еще больше.

[(Gurnee et al., 2023)](https://arxiv.org/abs/2305.01610) применили поиск к полноформатным языковым моделям, найдя интерпретацию для более чем 100 признаков в FFN-слоях, большинство из которых являются полисемантичными. При этом наблюдается четкая динамика по глубине сети: если в ранних слоях один нейрон кодирует множество (до 5–10 и более) несвязанных локальных паттернов и n-грамм (суперпозиция), то в средних слоях начинают доминировать узкоспециализированные, моносемантичные нейроны.

Объяснений два: признаки модели — это направления, не совпадающие с нейронами, и признаков больше, чем нейронов.

[(Park et al., 2023)](https://arxiv.org/abs/2311.03658) сформулировали гипотезу о линейных представлениях согласно которой признаки соответствуют не осям, а целым направлениям в пространстве активаций, а выраженность признака — проекции на это направление. Позже эта гипотеза была многократно подтверждена. Когда мы расмматривали Word2Vec эмбединги [(Mikolov et al., 2013)](https://aclanthology.org/N13-1090/), мы сталкивались с проявлением подобной линейной структуры пространства эмбедингов. Выражение $\vec{\text{король}}$ - $\vec{\text{мужчина}}$ + $\vec{\text{женщина}} \approx $ $\vec{\text{королева}}$ хорошо показывает, что конкретные признаки (например, пол) кодируются фиксированными направлениями.

Линейные пробы восстанавливают по активациям Llama-2 координаты городов и даты событий, так что по их предсказаниям можно нарисовать карту мира [(Gurnee & Tegmark, 2023)](https://arxiv.org/abs/2310.02207). 

[(Marks & Tegmark, 2023)](https://arxiv.org/abs/2310.06824) нашли в средних слоях направление, кодирующее истинность/ложность утвреждения ("Волга впадает в Каспийское море" => True). Что важно, тут речь не о правдоподобии наблюдаемой последовательности токенов и не о галлюцинациях - модель может заблуждаться. Речь именно об осознаваемом расхождении с правдой. Для поиска авторы взяли разницу центра масс настоящих и ложных фактов. Навык чтения намерений модели модели крайне важен для безопасности использования.

Бывает кодировка признаков и в подпростарнствах большей размерности, например, двумерной. Любопытно, что, как это показали [(Engels et al., 2024)](https://arxiv.org/abs/2405.14860), циклические данные, например, дни недели и месяцы в лежат на окружности в двумерной плоскости . То есть модель буквально строит в своем воображении циферблат.

Логика простая: компоненты читают поток линейными проекциями, поэтому нужный признак выгоднее всего сделать доступным одной проекцией.

Объясняли дистрибутивной гипотезой . [(Elhage et al., 2022)](https://arxiv.org/abs/2209.10652) предложили более современное объяснение - эффект который назвали суперпозицией, заимствовав аналогию с квантовой физикой. Идея в том, что когда понятий, которые нужны модели, заведомо больше, чем размерность эмбединга, . Но в многомерном пространстве почти ортогональных векторов помещается экспоненциально больше, чем измерений. Подвтерждили на контролируемым экспермиенте. Пока признаков мало, модель хранит только $m$ самых важных и игнорирует остальные. Как только становится много, она скачком переходит к суперпозиции и начинает упаковывает признаки в правильные фигуры: отрезки, треугольники, пятиугольники.

Интерференцию при этом отсекает ReLU с отрицательным смещением, поэтому без нелинейности суперпозиция невыгодна. Отсюда естественное объяснение полисемантичности: если признаков больше, чем нейронов, каждый нейрон вынужден участвовать в кодировании нескольких.

Почему вообще стоит смотреть на отдельные нейроны MLP, но не на отдельные координаты остаточного потока? Если повернуть поток ортогональной матрицей $R$, домножив на неё все читающие и пишущие матрицы, модель вычислит ту же самую функцию. Значит, у координат потока нет особого смысла, или, как говорят, у него нет привилегированного базиса (privileged basis). У нейронов MLP он есть: функция активации применяется покоординатно, и $\sigma(Rz) \neq R\,\sigma(z)$. На практике оптимизатор Adam всё же создаёт в потоке выделенные координаты с аномально большими значениями [(Elhage et al., 2023)](https://transformer-circuits.pub/2023/privileged-basis/index.html), но это скорее исключение.

## Наблюдательные методы

Дать что-то сделать м посмотреть под лупой, как модель его выполняет.

### Анализ нейронов
Самый прямой вариант анализа - посмотреть тепловую карту активаций.

[(Voita et al., 2023)](https://arxiv.org/abs/2309.04827) исследовали активации в архитекутуре моделей OPT. Несколько важных находок для понимания работы моделей. Во-первых, большинство нейронов ни разу не активировались. Особенно в ранних слоях (до 70% в отдельных слоях). то есть модель избыточна. Во-вторых, многие нейроны в ранних слоях работают как детекторы конкретных токенов или n-грамм, а заодно подавляют предсказание этих же токенов. В-третьих, нашли чисто позиционные нейроны, чья активация зависит только от позиции токена, а не от его содержания.

[(Dai et al., 2021)](https://arxiv.org/abs/2104.08696) обнаружили в BERT моделях наличие то, что они назвали нейронами "знания", которые находятся в FFN-слоях и отвечают за хранение фактические информации. Для этого они задавали вопросы на знание («Столица Франции — [MASK]») и выделяли нейроны, сильнее всего влияющие на правильный ответ (отключение портит ответ). Этот же принцип лег в основу техники точечного редактирования знаний в моделях - можно занулить нейрон, и модель забудет ассоциированный с ним факт.

Метод Dai правда быстро раскритиковали. В качестве контрпримера [(Niu et al., 2024)](https://arxiv.org/abs/2405.02421) замаскировали обычные слова, не факты, и FFN нейроны активировались на них ничуть не меньше. То есть на самом деле нейроны реагировали больше на синтаксическую согласованность, чем на необходимость достать конкретный факт. Кроме того, редактирование через зануление этих нейронов работает слабо. Фактическая информация действительно хранится в FFN, но не концентрировано в отдельных нейронах а все таки распределённо.

Также были попытки по карте акиваций выделить общий паттерн - смотрим, когда нейрон сильнее всего срабатывает, определяем признак и ассоциируем с ним. Её также подвергли критике. []() провели эксперимент, на нескольких корпусах текстов. На каждом корпусе это были совершенно разные запросы.

### Пробинг
Искать паттерны, но большее распространение получил противоположный подход - по паттерну.

Мы выбираем некоторое свойство (например, "тональность высказывания" или "пол главного субъекта"), поверх активаций анализиуруемого нейрона присоединить небольшой классификатор. Веса модели замораживаются, а этот классификатор обучается определять наличие данного свойства. [(Alain & Bengio, 2016)](https://arxiv.org/abs/1610.01644) назвали такой классификатор зондом (Probe). Высокая точность пробинга означает, что сигнал в активациях есть. Применять пробинг можно как к отдельному токену, или к фрагменту текста - тогда делается усредение.

Перебирая различные свойства, можно построить довольно детальную карту, что где происходит в Трансформере. Так [(Tenney et al., 2019)](https://arxiv.org/abs/1905.05950) выяснили, что в BERT моделях стабильно наблюдается четкая иерархия признаков: части речи распознаются в нижних слоях, синтаксис в средних, а сущностные связи (coreferences) в самых верхних. [(Hewitt & Manning, 2019)](https://aclanthology.org/N19-1419/) выяснили, что по активациям можно даже восстанавливать синтаксические деревья - чем ближе токены в дереве, тем ближе их активации.

У пробинга есть две проблемы. Во-первых, слишком многослойный классификатор может выучить задачу, поэтому её сравнивают с контрольной задачей со случайными метками и смотрят на разницу точностей [(Hewitt & Liang, 2019)](https://arxiv.org/abs/1909.03368). Во-вторых, пробинг детектирует наличие сигнала, но не его фактическое использование для генерации ответа. Чтобы проверить, нужна ли информация модели, нейроны зануляют и смотрят: как меняется точность предсказания. [(Elazar et al., 2020)](https://arxiv.org/abs/2006.00995) такую механику назвали амнезический пробинг.

### Logit Lens
Идея метода - подать активации с каждого промежуточного слоя сразу на последнюю проекцию и посмотреть, как юудет меняться выходное распределение. [(Nostalgebraist, 2020)](https://www.lesswrong.com/posts/AcKRB8wDpdaN6v6ru/interpreting-gpt-the-logit-lens) назвали этот приём Logit Lens, логиты анализируются под микроскопом. На примере модели GPT-2 авторы показали, как прогноз постепенно уточняется по слоям, а для простых токенов складывается уже к середине сети. 

Проблема метода в том что представления промежуточных слоёв «дрейфуют», и logit lens выдаёт шум. Модификация метода Tuned lens дополниетльно обучая для каждого слоя небольшое аффинное преобразование в пространство последнего слоя [(Belrose et al., 2023)](https://arxiv.org/abs/2303.08112). С помощью Logit Lens, например, [(Wendler et al., 2024)](https://arxiv.org/abs/2402.10588) выяснили, что модель Llama-2 при переводе с французского на китайский в средних слоях "думает" английскими словами.

### Паттерны внимания
Веса внимания соблазнительно считать объяснением: их легко нарисовать, а в BERT нашлись головы, отслеживающие синтаксические отношения [(Clark et al., 2019)](https://arxiv.org/abs/1906.04341). Но вокруг них развернулся спор. [(Jain & Wallace, 2019)](https://arxiv.org/abs/1902.10186) показали, что совсем другие распределения внимания часто дают тот же ответ, а [(Wiegreffe & Pinter, 2019)](https://arxiv.org/abs/1908.04626) возразили, что вывод зависит от того, что считать объяснением. С механистической точки зрения ограничения понятны. Паттерн описывает только QK-схему и молчит о том, что голова переносит: большой вес на служебном токене обычно сопровождается почти нулевым вкладом [(Kobayashi et al., 2020)](https://arxiv.org/abs/2004.10102). Первый токен часто служит «стоком» внимания, куда голова сбрасывает вес, когда ей нечего искать [(Xiao et al., 2023)](https://arxiv.org/abs/2309.17453). Поэтому паттерны хороши для гипотез, но не для доказательств.

### Прямая атрибуция к логитам
Раз остаточный поток — сумма вкладов, логит можно разложить по компонентам. Заморозив нормализацию с коэффициентом $\sigma$, получим вклад компонента $c$ в выбор токена $t_1$ вместо $t_2$:

$$\mathrm{DLA}(c) = \frac{\big(W_U[t_1] - W_U[t_2]\big) \cdot c}{\sigma}$$

Например, в предложении «Когда Мэри и Джон пошли в магазин, Джон дал напиток» (в оригинале на английском) так находят головы, которые сильнее всего толкают модель к ответу «Мэри». Прямая атрибуция (direct logit attribution) считается за один прогон, но видит только прямой эффект: голова, которая направляет внимание другой головы, но сама в логиты не пишет, останется незамеченной. Чтобы измерить полный эффект, нужно вмешиваться.

## Каузальные интервенции
Что будет, если покрутить параметр модели? Сильно ли изменится выход?

Корреляция не равна причинности. Наблюдательные методы типа пробинга могут показать наличие в активациях нужной информации, но не объясняет, пользуются ли ей модель. Для этого нужен тест на причинность - запустить два варианта (оригинальный и исправленный) и сравнить результат. Для нейросети вмешательство дёшево: можно заменить любую активацию и повторить прогон — о таком нейробиолог может только мечтать. 

Одними из первых это систематически сделали [(Vig et al., 2020)](https://arxiv.org/abs/2004.12265). Они перенесли из эпидемиологии каузальный анализ медиации (causal mediation analysis) и применили его к GPT-2. Идея такая: меняем вход $x$ на $x'$ и сравниваем полный эффект на выход с той его частью, которая проходит через отдельный компонент $z$:

$$
\mathrm{TE} = M(x') - M(x), \qquad \mathrm{IE}_z = M\big(x;\; z \leftarrow z(x')\big) - M(x)
$$

Здесь $M$ — интересующая нас метрика, например вероятность правильного ответа. __Полный эффект__ (total effect) показывает, насколько изменится метрика, если поменять вход целиком. __Косвенный эффект__ (indirect effect) — насколько она изменится, если вход оставить прежним, а компоненту $z$ подставить то значение, которое он принял бы на $x'$. Если $\mathrm{IE}_z$ близок к $\mathrm{TE}$, значит почти всё влияние идёт через $z$. Все методы ниже — вариации этой идеи.

Самое простое вмешательство — __абляция__ (ablation): выключить компонент и посмотреть, как изменится метрика. Вопрос в том, что значит «выключить». Активацию можно заменить нулём, средним значением по датасету или значением с другого случайного примера. [(Li & Janson, 2024)](https://arxiv.org/abs/2409.09951) предложили ещё один вариант — константу, которая минимизирует потери модели.

Есть и вторая сложность: модели умеют себя чинить. [(McGrath et al., 2023)](https://arxiv.org/abs/2307.15771) обнаружили, что если выключить слой внимания, последующие слои берут часть его функций. Явление назвали эффектом гидры (hydra effect) — на месте отрубленной головы вырастает новая. Это значит, что величина эффекта абляции быть заниженой.

Абляция отвечает на вопрос, важен ли компонент вообще. __Патчинг активаций__ (activation patching, или causal tracing) отвечает на более точный: через какие компоненты проходит конкретная информация. Берут пару минимально различающихся входов: чистый (clean), который вызывает интересующее поведение, и испорченный (corrupted), который его не вызывает. Например, «Когда Мэри и Джон пошли в магазин, Джон дал напиток» → «Мэри», и тот же текст с тремя другими именами. Прогоняем испорченный вход, но в одном месте подставляем активацию из чистого прогона. Если поведение восстановилось, значит нужная информация проходит через это место. Перебрав слои, позиции и компоненты, получаем карту важности.

<img src="img/patching.png" width=550>

Восстановление нужно чем-то измерять. [(Zhang & Nanda, 2023)](https://arxiv.org/abs/2309.16042) рекомендуют не вероятность ответа, а разность логитов правильного и неправильного ответа $\mathrm{LD} = \mathrm{logit}_{\text{correct}} - \mathrm{logit}_{\text{wrong}}$. Вероятность — нелинейная функция логитов и насыщается: если в чистом прогоне она уже 0.99, то компонент, сдвигающий логит ещё на несколько единиц, почти ничего не изменит, и его вклад окажется незаметен. Логиты же получаются линейной проекцией последнего скрытого состояния, поэтому вклады компонентов в $\mathrm{LD}$ примерно складываются. Чтобы результаты разных задач были сравнимы, разность нормируют на её значения в чистом и испорченном прогонах:

$$
\mathrm{score} = \frac{\mathrm{LD}_{\text{patched}} - \mathrm{LD}_{\text{corr}}}{\mathrm{LD}_{\text{clean}} - \mathrm{LD}_{\text{corr}}}
$$

Ноль означает, что патч ничего не восстановил, единица — что поведение восстановилось полностью.

Метод прославила работа ROME [(Meng et al., 2022)](https://arxiv.org/abs/2202.05262). Авторы брали промпт «Эйфелева башня находится в городе» и портили его, зашумляя эмбеддинги субъекта. Патчинг показал, что факт достают MLP средних слоёв на последнем токене субъекта, а внимание верхних слоёв переносит его к позиции ответа. На этой основе они предложили редактировать факты точечным изменением весов одного слоя MLP. Продолжение оказалось поучительным: [(Hase et al., 2023)](https://arxiv.org/abs/2301.04213) показали, что редактирование слоёв, где факт якобы не хранится, работает не хуже. Локализация в смысле патчинга и «место, где удобно вмешаться», — разные вещи.

Важно и направление патчинга, на что обратили внимание [(Heimersheim & Nanda, 2024)](https://arxiv.org/abs/2404.15255). Подставляя чистые активации в испорченный прогон (denoising), мы ищем компоненты, которых __достаточно__ для поведения. Подставляя испорченные активации в чистый прогон (noising), ищем __необходимые__. Результаты могут расходиться. Если ответ требует одновременной работы двух голов (логическое И), денойзинг по отдельности не найдёт ни одной из них: одной головы недостаточно. Если хватает любой из двух (логическое ИЛИ, например основная и запасная), ни одной не найдёт нойзинг: каждая по отдельности не необходима. Поэтому используют оба варианта.

Обычный патчинг измеряет полный эффект компонента — всё его влияние через все последующие слои. Чтобы восстановить схему (circuit), нужно знать связи: какая голова влияет на какую. Для этого [(Goldowsky-Dill et al., 2023)](https://arxiv.org/abs/2304.05969) предложили __патчинг путей__ (path patching): изменение пропускается только по ребру от компонента-отправителя к компоненту-получателю, а все остальные пути замораживаются.

Оба метода дороги: на каждое место нужен отдельный прогон, а мест — слои × позиции × головы, то есть тысячи. Нил Нанда предложил заменить патчинг линейным приближением — __атрибуционным патчингом__ (attribution patching) [(Nanda, 2023)](https://www.neelnanda.io/mechanistic-interpretability/attribution-patching). Это разложение Тейлора первого порядка: изменение метрики примерно равно скалярному произведению изменения активации на градиент метрики по ней.

$$
M\big(x_{\text{corr}};\; a \leftarrow a_{\text{clean}}\big) - M(x_{\text{corr}}) \approx \big(a_{\text{clean}} - a_{\text{corr}}\big)^\top \left.\frac{\partial M}{\partial a}\right|_{x_{\text{corr}}}
$$

Выигрыш в том, что градиент по всем активациям сразу даёт один обратный проход. Итого для всей карты хватает двух прямых прогонов (чистого и испорченного) и одного обратного. Приближение ломается там, где модель резко нелинейна. Например, при насыщенном softmax внимания градиент почти нулевой, хотя настоящий патч может полностью перестроить паттерн внимания. [(Kramár et al., 2024)](https://arxiv.org/abs/2403.00745) предложили улучшенную версию AtP*, которая исправляет основные такие случаи. На практике атрибуционный патчинг используют как дешёвый фильтр, а отобранных кандидатов перепроверяют обычным патчингом.

Конечная цель всего этого — не список важных компонентов, а алгоритм. [(Geiger et al., 2021)](https://arxiv.org/abs/2106.02997) предложили теорию __каузальных абстракций__ (causal abstraction). Гипотеза формулируется как маленький каузальный граф с понятными переменными, и проверяется, что вмешательство в соответствующее представление модели действует так же, как вмешательство в переменную графа. Пусть гипотеза гласит, что при сложении $a + b + c$ модель сначала вычисляет $s = a + b$. Подставим представление $s$ из прогона на $4 + 3 + 9$ в прогон на $1 + 2 + 5$. Если гипотеза верна, модель ответит $7 + 5 = 12$ — не 8, как на исходном входе, и не 16, как на входе-доноре.

Сложность в том, что переменная $s$ не обязана жить в отдельных нейронах — она может быть распределена по подпространству. Метод DAS (distributed alignment search) от [(Geiger et al., 2023)](https://arxiv.org/abs/2303.02536) обучает ортогональный поворот $R$ и вмешивается только в первые $k$ координат повёрнутого базиса:

$$
\tilde{h} = R^\top \big(P_k R\, h_{\text{src}} + (I - P_k) R\, h\big)
$$

где $P_k$ — проектор на первые $k$ координат, $h$ — активация исходного прогона, $h_{\text{src}}$ — прогона-донора. Поворот $R$ подбирают так, чтобы ответы модели после вмешательства совпадали с предсказаниями гипотезы. По сути это пробинг, но с каузальным критерием качества.

Гибкость здесь опасна. [(Makelov et al., 2023)](https://arxiv.org/abs/2311.17030) показали, что найденное подпространство может «будить» спящий путь, который в обычной работе модели не участвует: вмешательство работает, но гипотеза при этом неверна. Ещё строже проверяет гипотезы метод __causal scrubbing__ [(Chan et al., 2022)](https://www.lesswrong.com/posts/JvZhhzycHu2Yd57RN/causal-scrubbing-a-method-for-rigorously-testing): всё, что гипотеза объявляет неважным, заменяется ресэмплингом. Если гипотеза полна, качество модели не падает.

## Цепи нейронов
Схема (circuit) — это цепочка связанных нейронов в нейронной сети, которая отвечает за активацию конкретного паттерн. Если предыдущие методы выполняли анализ на уровне отдельного нейрона, то мы ищем нейронов.

### Индукционные головы
Решили посмотреть, как именно работает внимание в специфических ситуациях генерации проолжения.
Первым кейсом стал In-context llearning. Продолжение при начлии конектста. подробно разобранной схемой в Трансформерах стали индукционные головы [(Olsson et al., 2022)](https://arxiv.org/abs/2209.11895). Если в тексте уже встречалась пара токенов $[a][b]$, то, увидев $a$ снова, модель предсказывает $b$: «В книге Анна Каренина. Имя героини Анна ...» → «Каренина».

Оказывалось, что у голов самовнимания появляется специализация. Авторы выделяют два типа голов. Первый назвали previous token head, которая работает просто как функция `lag()` - она запоминает предыдуший токен. Второй - induction head, которая ищет в предыдщуем тексте тот же префикс A и запоминает его разрешение B. Это разрешегние B и идет в продолжение.

Любопытно, что индукционные головы появляются скачком: в определённый момент обучения на кривой потерь видна ступенька, и одновременно резко улучшается обучение в контексте (in-context learning). Авторы приводят ряд аргументов в пользу того, что индукционные головы — один из главных механизмов этой способности.

[рисунок: паттерн внимания индукционной головы на повторяющейся последовательности]

### Схема IOI
Далее [(Wang et al., 2022)](https://arxiv.org/abs/2211.00593) решили специфицировать задачу и посомтреть, как модель выбирает косвенное дополнение задача IOI (indirect object identification). В предложении «Когда Мэри и Джон пошли в магазин, Джон дал напиток ...» модель должна назвать Мэри, не Джон.

Использовали GPT-2 с 144 головами самовнимания. Удалось найти схему из 26 голов и имеющих узкую специализацию. К двум головам, которые уже рассматривали, "Previous Token head" и "Induction Head" добавились еще 4. Головы дубликатов (Duplicate Token head) замечают, что «Джон» уже встречался. Головы S-ингибиции (Subject Inhibition) подавляют внимание к повторившемуся имени. Головы перемещения имён (name movers) смотрят на оставшееся имя и копируют его в ответ. Нашлись и запасные головы, которые берут работу на себя, если выключить основные, и головы, которые, наоборот, снижают уверенность модели.

### Другие схемы
[(Hanna et al., 2023)](https://arxiv.org/abs/2305.00586) разобрали, как GPT-2 продолжает фразу «Война длилась с 1732 по 17..». Внимание переключается на токен "32". А нейроны в поздних MLP слоях моделируют сравнение. Что любопытно, вероятность ответа неравномерна, то есть модель знает типовую длительность события. 

[(Nanda et al., 2023)](https://arxiv.org/abs/2301.05217) исследовали как Трансформер, выучившего сложение по модулю: числа представлялись поворотами на окружности, а сложение выполнялось через тригонометрические тождества (эту работу мы упоминали в главе 5 в связи с гроккингом). [(Lieberum et al., 2023)](https://arxiv.org/abs/2307.09458) проверили, масштабируется ли анализ схем, на модели Chinchilla с 70 млрд параметров: ключевые головы для ответа на вопрос с вариантами найти удалось, но их объяснение лишь частично переносилось на другие формулировки задачи.

### Автоматический поиск схем
Ручной поиск подходящих схем трудоемкий. Метод ACDC [(Conmy et al., 2023)](https://arxiv.org/abs/2304.14997) автоматизирует его: идёт от выхода ко входу и удаляет из графа рёбра, патчинг которых почти не меняет метрику. Edge attribution patching [(Syed et al., 2023)](https://arxiv.org/abs/2310.10348) заменяет перебор градиентным приближением и работает на порядки быстрее. Сравнивать методы поиска схем помогает бенчмарк MIB [(Mueller et al., 2025)](https://arxiv.org/abs/2504.13151).

У анализа схем есть ограничения. Каждую схему ищут под конкретную задачу и конкретный набор промптов. Самовосстановление искажает оценки важности. А главное, на уровне голов и нейронов схемы получаются запутанными из-за суперпозиции. Следующий шаг — перейти от компонентов к признакам.

## Разреженные автоэнкодеры
Автоэнкодеры - очень старая механика, идея которой появилась ещё в конце 80-х. Целью является поиск сжатого внутреннего преставления входного образа, такого, чтобы он содержал максимум полезного сигнала. Представление считается полезным, если из него обучаемым преобразованием можно восстановить исходный сигнал с минимальной ошибкой.

$$z = \sigma(W_{enc}\, x), \qquad \hat{x} = W_{dec}\, z, \qquad \mathcal{L} = \lVert x - \hat{x} \rVert^2, \qquad \dim z < \dim x
$$

<img src="img/ae.png" width=550>

Разреженные автоэнкодеры (sparse autoencoder, SAE) - появвившаяся в конце 2000-х модфификация, когда проекция в промежуточный слой не сжимающая, а расширяющая, и на промежуточное преставление $f$ накладывается ограничение на разреженность, обычно в виде штрафа. Так, входной сигнал проецируется в few-hot представление. Эту задачу также называют задачей словарного обучения (dictionary learning).

$$
f = \mathrm{ReLU}(W_{enc}\, x), \qquad \hat{x} = W_{dec}\, f, \qquad \mathcal{L} = \lVert x - \hat{x} \rVert^2 + \lambda \lVert f \rVert_1, \qquad \dim f \gg \dim x
$$

В 2023 году [(Cunningham et al., 2023)](https://arxiv.org/abs/2309.08600) применили разреженный автоэнкодер к активациям языковых моделей и показали, что признаки неплохо интерпретируются. Так например, на однослойной модели удалось выделить признаки арабского письма, последовательностей ДНК, кодировки base64 и тысячи других.

<img src="img/sae.png" width=550>

Для штрафа вместо нормы $L_0$ используют его гладкую замену $L_1$. Она работает не идеально - зануляя слабые признаки, она также занижает сильные (эффект Lasso). Пусть мы минимизируем функционал $(a - f)^2 + \lambda f$, где a признак, тогда минимум равен $f^* = \max(0,\, a - \lambda/2)$. Видим, что признаки с $a < \lambda/2$ обнуляются целиком, а активные признаки занижаются на $\lambda/2$.

Поэтому существуют другие архитектурные варианты. Самое простое - оставлят топ $k$ наибольших активаций, как это делали [(Gao et al., 2024)](https://arxiv.org/abs/2406.04093) в OpenAI. Либо метод JumpReLU от [(Rajamanoharan et al., 2024)](https://arxiv.org/abs/2407.14435), где у каждого компонента есть обучаемый порог.

Эффективность разреженного автоэнкодинга оценивают сразу несколькими метриками. Во-первых, по степени разреженнгости, то бишь среднему числу ненулевых компонентов в векторе $f$. Во-вторых, по доле объяснённой дисперсии - считается $R^2$ точности пронозного x, только векторный. В третьих, по росту функции потерь модели при замене входа $x$ его разреженным приближением $\hat{x}$. И наконец по интерпретируемости получаемых признаков - оценкой занимается другая языковая модель. Все эти метрики сводит вместе бенчмарк SAEBench от [(Karvonen et al., 2025)](https://arxiv.org/abs/2503.09532).

Как отмечают [(Bricken et al., 2023)](https://transformer-circuits.pub/2023/monosemantic-features/index.html), гранулярность признаков - вещь динамическая, с ростом словаря признаки становится более детальными (feature splitting): «математика» распадается на «алгебру», «геометрию» и так далее. Бывает и другое явление, поглощение признака (feature absorption), которое описали [(Chanin et al., 2024)](https://arxiv.org/abs/2409.14507). Они обратили внимание, что при коллизии двух признаков: общего и специфичного (например, «слово начинается на S» и «слово short») включается только один - специфичный, чтобы не нарушить разреженность.

В 2024 году автоэнкодерную интерпретацию применили к моделям большего размера. [(Templeton et al., 2024)](https://transformer-circuits.pub/2024/scaling-monosemanticity/index.html) из Anthropic проанализировали Claude 3 Sonnet и построили вектор из 34 млн признаков. Они оказались более абстрактными, мультиязычными и мультимодальными: признак моста Golden Gate активировался и на упоминания моста на разных языках, и на его фотографии. Существенно то, что нашлись и признаки, важные для безопасности: готовность к обману, угодничество, уязвимый код.

До этого автоэнкодер использовали чтобы распутать промежуточный сигнал. [(Dunefsky et al., 2024)](https://arxiv.org/abs/2406.11944) предложили __транскодер__ - версию автоэнкодера, который обучается под немного другую задачу - он восстанавливает не исходный сигнал $x$, а выход $f(x)$ некоторой трансформации, например, FFN слоя. То есть декомпзируется не состояние, а преобразвание. 

Другая модификация __Кросскодер__ (crosscoder) читает и восстанавливает сразу несколько слоёв или даже две модели, что позволяет, например, найти признаки, появившиеся при дообучении базовой модели в чат-модель [(Lindsey et al., 2024)](https://transformer-circuits.pub/2024/crosscoders/index.html).

Подход не является серебряной пулей. Во-первых, автоэнкодинг восстанавливает далеко не всё, есть часть, которая не описывается, и которую [(Engels, Riggs & Tegmark, 2024)](https://arxiv.org/abs/2410.14670) назвали "тёмной материей". Она прогнозируется, и в ней может содержаться важный сигнал. Во-вторых, состав признаков сильно зависит от размера модели и словаря. Наконец, на практических задачах SAE пока часто проигрывают простые методы не уступают: например, [(Kantamneni et al., 2025)](https://arxiv.org/abs/2502.16681) показали, что линейные пробы на сырых активациях, не уступают по точности пробам на разреженных признаках.

## Графы атрибуции
Следующий шаг — соединить признаки в графы вычислений. Предшественником стала работа о разреженных схемах признаков [(Marks et al., 2024)](https://arxiv.org/abs/2403.19647), а в 2025 году Anthropic предложила метод отслеживания схем (circuit tracing) [(Ameisen et al., 2025)](https://transformer-circuits.pub/2025/attribution-graphs/methods.html). Все MLP модели заменяются межслойными транскодерами, которые пишут сразу в несколько последующих слоёв, и получается замещающая модель (replacement model), где вычисление проходит через интерпретируемые признаки. Для конкретного промпта паттерны внимания и нормализацию фиксируют, после чего связи между признаками становятся линейными, и вклад каждого признака в каждый следующий считается точно. Получается граф атрибуции (attribution graph): узлы — активные признаки, рёбра — их влияния друг на друга. Ошибку транскодеров добавляют в граф отдельными узлами, чтобы честно показать, какая часть вычислений осталась необъяснённой.

[рисунок: граф атрибуции для вопроса о столице штата, где находится Даллас]

С помощью таких графов авторы разобрали десяток поведений модели Claude 3.5 Haiku [(Lindsey et al., 2025)](https://transformer-circuits.pub/2025/attribution-graphs/biology.html). На вопрос о столице штата, где находится Даллас, модель сначала активирует признак «Техас» и уже из него получает «Остин», то есть действительно рассуждает в два шага. Сочиняя рифмованную строку, модель заранее выбирает слово для рифмы и строит строку так, чтобы к нему прийти; если подавить этот признак, строка заканчивается другой рифмой. При сложении двузначных чисел работают два параллельных пути: один грубо оценивает величину суммы, другой точно вычисляет последнюю цифру. При этом, если спросить модель, как она считала, она описывает школьный алгоритм с переносом. Галлюцинации объясняются так: по умолчанию в модели активна схема «я не знаю ответа», а признаки знакомой сущности её подавляют. Если имя кажется знакомым, но фактов о нём модель не знает, подавление срабатывает ошибочно.

Ограничения метода авторы перечисляют сами: граф описывает один промпт, объясняет лишь часть вычислений и плохо показывает, почему внимание смотрит туда, куда смотрит. Инструменты для построения графов на открытых моделях опубликованы в библиотеке circuit-tracer вместе с интерфейсом на Neuronpedia [(Anthropic, 2025a)](https://www.anthropic.com/research/open-source-circuit-tracing).

## Управление через представления
Если признаки — это направления, их можно не только читать, но и добавлять. Самый простой способ получить такое направление — взять разность средних активаций на двух контрастных наборах промптов и добавить её в остаточный поток с коэффициентом $\alpha$:

$$v = \mathbb{E}\big[h_l(x) \mid x \in \mathcal{D}^{+}\big] - \mathbb{E}\big[h_l(x) \mid x \in \mathcal{D}^{-}\big], \qquad h_l \leftarrow h_l + \alpha\, v$$

Вектор $v$ называют вектором управления (steering vector). Уже разность активаций для слов «Love» и «Hate», добавленная в поток, меняет тональность генерации [(Turner et al., 2023)](https://arxiv.org/abs/2308.10248). Метод CAA усредняет такие разности по многим парам ответов и управляет, например, угодничеством модели [(Rimsky et al., 2023)](https://arxiv.org/abs/2312.06681). Метод ITI находит головы внимания, связанные с правдивостью, и сдвигает их активации, повышая долю правдивых ответов [(Li et al., 2023)](https://arxiv.org/abs/2306.03341). Общий подход к чтению и управлению концептами вроде честности или эмоций назвали representation engineering [(Zou et al., 2023)](https://arxiv.org/abs/2310.01405).

Самый наглядный результат касается безопасности. [(Arditi et al., 2024)](https://arxiv.org/abs/2406.11717) показали, что отказ отвечать на вредные запросы в 13 открытых чат-моделях опосредован одним направлением. Если вычесть проекцию на него, модель перестаёт отказываться, а если добавить, отказывается даже на безобидные просьбы. Это говорит о том, что выравнивание через дообучение может оказаться довольно тонким слоем поверх базовой модели.

Управлять можно и через признаки SAE: если зафиксировать признак моста Golden Gate на большом значении, Claude начинает считать себя этим мостом [(Templeton et al., 2024)](https://transformer-circuits.pub/2024/scaling-monosemanticity/index.html). Векторы персоны (persona vectors) позволяют отслеживать черты вроде угодничества или склонности к галлюцинациям, в том числе в ходе дообучения, и находить обучающие данные, которые эти черты усиливают [(Chen et al., 2025)](https://arxiv.org/abs/2507.21509).

Насколько всё это практично? Бенчмарк AxBench [(Wu et al., 2025)](https://arxiv.org/abs/2501.17148) сравнил методы управления и показал, что обычный промптинг пока выигрывает у методов на основе представлений, а признаки SAE уступают даже простой разности средних. Поэтому управление через представления остаётся скорее исследовательским инструментом и способом проверить гипотезу о признаке, чем заменой промптинга или дообучения.

## Масштабирование интерпретации
В большой модели миллионы признаков, и описать их вручную невозможно. [(Bills et al., 2023)](https://openaipublic.blob.core.windows.net/neuron-explainer/paper/index.html) поручили это другой модели: GPT-4 получала тексты с активациями нейрона GPT-2 и писала объяснение. Качество объяснения проверяли симуляцией: модель по одному объяснению предсказывала активации на новых текстах, и корреляция с реальными активациями служила оценкой. Хорошо объяснить удалось лишь небольшую долю нейронов, что неудивительно при полисемантичности. С признаками SAE дело идёт лучше, и сегодня автоматические описания строят для миллионов признаков сразу [(Paulo et al., 2024)](https://arxiv.org/abs/2410.13928).

Следующий шаг — агенты-интерпретаторы, которые сами ставят эксперименты: подбирают входы, вмешиваются в активации и уточняют гипотезы, как это делает MAIA [(Shaham et al., 2024)](https://arxiv.org/abs/2404.14394). Anthropic испытала агентов, которые с помощью инструментов интерпретации проводят аудит моделей и ищут у них скрытые нежелательные цели [(Anthropic, 2025b)](https://alignment.anthropic.com/2025/automated-auditing). Главный риск тот же, что и у людей: объяснение может звучать убедительно и при этом быть неверным [(Huang et al., 2023)](https://arxiv.org/abs/2309.10312). Поэтому автоматические описания тоже нужно проверять вмешательствами.

## Критика
В интерпретируемости легко найти то, что ищешь. Хорошая иллюстрация — эксперимент [(Jonas & Kording, 2017)](https://journals.plos.org/ploscompbiol/article?id=10.1371/journal.pcbi.1005268). Авторы применили стандартные методы нейробиологии к процессору MOS 6502, на котором запускали старые видеоигры. Методы выдавали правдоподобные результаты, например нашлись транзисторы, удаление которых ломало только Donkey Kong, но ничего не говорили о реальном устройстве процессора.

Типичные ошибки мы уже встречали: объяснение по отобранным примерам, которое не переносится на другой датасет; патчинг подпространств, который «будит» неиспользуемые пути; локализация, которая не подсказывает, где вмешиваться. Отсюда несколько правил. Проверять каждую гипотезу вмешательством, а не только наблюдением. Сравнивать результат со случайными направлениями и случайными компонентами. Формулировать гипотезы так, чтобы они предсказывали поведение на новых входах. И измерять, какую долю поведения объясняет найденный механизм. Для сравнения методов появились бенчмарки: MIB для схем и каузальных переменных [(Mueller et al., 2025)](https://arxiv.org/abs/2504.13151) и SAEBench для автоэнкодеров.

Открытых проблем много [(Sharkey et al., 2025)](https://arxiv.org/abs/2501.16496). Неясно, какая единица анализа правильная: нейрон, направление, признак SAE или что-то ещё. Методы плохо масштабируются, и даже лучшие графы атрибуции объясняют лишь часть вычислений. Мало известно о том, как механизмы формируются в процессе обучения; этим занимается отдельное направление, developmental interpretability [(Hoogland et al., 2024)](https://arxiv.org/abs/2402.02364). И пока нет способа строго доказать, что модель чего-то не делает.

## Применения
Главная мотивация многих исследований — безопасность. В эксперименте [(Marks et al., 2025)](https://arxiv.org/abs/2503.10965) модель намеренно обучили скрытой цели: подстраиваться под известные ошибки модели вознаграждения. Затем несколько команд, не знавших об этом, провели аудит, и команды с доступом к весам и признакам SAE нашли причину странного поведения. Линейные пробы на активациях Llama-3.3-70B распознают стратегический обман, например в сценарии с инсайдерской торговлей [(Goldowsky-Dill et al., 2025)](https://arxiv.org/abs/2502.03407). А векторы персоны из предыдущего раздела позволяют заметить нежелательный сдвиг характера модели ещё во время дообучения.

Интерпретируемость помогает и в отладке. Метод SHIFT из работы о разреженных схемах признаков [(Marks et al., 2024)](https://arxiv.org/abs/2403.19647) нашёл в классификаторе профессий признаки, связанные с полом, и, выключив их, убрал ложную корреляцию без сбора новых данных. Редактирование знаний (ROME и MEMIT [(Meng et al., 2022)](https://arxiv.org/abs/2210.07229)) и удаление опасных знаний, например методом RMU [(Li et al., 2024)](https://arxiv.org/abs/2403.03218), тоже опираются на локализацию информации внутри модели, хотя, как мы видели, связь локализации с удачным вмешательством неочевидна.

Наконец, интерпретируемость может учить самих людей. [(Schut et al., 2023)](https://arxiv.org/abs/2310.16410) извлекли из AlphaZero шахматные концепции, которых нет в человеческой теории игры, и показали, что гроссмейстеры способны им научиться. Похожие работы ведутся на моделях белков и геномов, где модель может знать то, чего ещё не знает биология.

Механистическая интерпретируемость остаётся молодой областью: многие её результаты получены на небольших моделях, а методы меняются каждый год. Но уже сейчас она позволяет задавать вопросы, которые раньше казались невозможными: не только что модель отвечает, но и почему.